In [0]:
# ============================================
# Part 1: Load and Inspect the Data
# Databricks / PySpark Code
# ============================================

from pyspark.sql import SparkSession

# 1. Start Spark Session
spark = SparkSession.builder \
    .appName("Ecommerce_Data_Analysis") \
    .getOrCreate()


In [0]:
# ============================================
# 2. Read CSV Files into Spark DataFrames
# ============================================

users_df = spark.read.csv(
    "/Volumes/workspace/default/ecommerce/users.csv",
    header=True,
    inferSchema=True
)

orders_df = spark.read.csv(
    "/Volumes/workspace/default/ecommerce/orders.csv",
    header=True,
    inferSchema=True
)

order_items_df = spark.read.csv(
    "/Volumes/workspace/default/ecommerce/order_items.csv",
    header=True,
    inferSchema=True
)

products_df = spark.read.csv(
    "/Volumes/workspace/default/ecommerce/products.csv",
    header=True,
    inferSchema=True
)



In [0]:
# ============================================
# 3. Display First 5 Rows
# ============================================

print("===== USERS DATA =====")
users_df.show(5)

print("===== ORDERS DATA =====")
orders_df.show(5)

print("===== ORDER ITEMS DATA =====")
order_items_df.show(5)

print("===== PRODUCTS DATA =====")
products_df.show(5)


===== USERS DATA =====
+-----+----------+---------+--------------------+---+------+-----+--------------------+-----------+-----------+-------+------------+------------+--------------+-------------------+
|   id|first_name|last_name|               email|age|gender|state|      street_address|postal_code|       city|country|    latitude|   longitude|traffic_source|         created_at|
+-----+----------+---------+--------------------+---+------+-----+--------------------+-----------+-----------+-------+------------+------------+--------------+-------------------+
|44262|   Michael|  Sanchez|michaelsanchez@ex...| 48|     M|  Mie|     5379 Kim Corner|   513-0836|Suzuka City|  Japan| 34.85181443| 136.5087133|      Facebook|2020-12-05 14:39:00|
|61852|     David|   Watson|davidwatson@examp...| 21|     M|  Mie|58568 Brooks Plai...|   513-0836|Suzuka City|  Japan| 34.85181443| 136.5087133|        Search|2022-01-24 13:00:00|
|82418|      Lisa|   Rivera|lisarivera@exampl...| 29|     F|  Mie| 3092 

In [0]:
# ============================================
# 4. Print Schema of Each DataFrame
# ============================================

print("===== USERS SCHEMA =====")
users_df.printSchema()

print("===== ORDERS SCHEMA =====")
orders_df.printSchema()

print("===== ORDER ITEMS SCHEMA =====")
order_items_df.printSchema()

print("===== PRODUCTS SCHEMA =====")
products_df.printSchema()

===== USERS SCHEMA =====
root
 |-- id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- state: string (nullable = true)
 |-- street_address: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- traffic_source: string (nullable = true)
 |-- created_at: timestamp (nullable = true)

===== ORDERS SCHEMA =====
root
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- returned_at: timestamp (nullable = true)
 |-- shipped_at: timestamp (nullable = true)
 |-- delivered_at: timestamp (nullable = tr

In [0]:
# ============================================
# 5. Find Row Count of Each DataFrame
# ============================================

print("===== ROW COUNTS =====")

print("Users Count:", users_df.count())
print("Orders Count:", orders_df.count())
print("Order Items Count:", order_items_df.count())
print("Products Count:", products_df.count())

===== ROW COUNTS =====
Users Count: 100000
Orders Count: 124923
Order Items Count: 180952
Products Count: 29120


In [0]:
# ============================================
# 6. Identify Join Columns
# ============================================

print("===== PRIMARY JOIN KEYS =====")

join_keys = {
    "users.csv": ["id"],
    "orders.csv": ["order_id", "user_id"],
    "order_items.csv": ["order_id", "user_id", "product_id"],
    "products.csv": ["id"]
}

for table, keys in join_keys.items():
    print(f"{table} --> {keys}")


===== PRIMARY JOIN KEYS =====
users.csv --> ['id']
orders.csv --> ['order_id', 'user_id']
order_items.csv --> ['order_id', 'user_id', 'product_id']
products.csv --> ['id']


In [0]:
# ============================================
# Suggested Joins
# ============================================

print("\n===== SUGGESTED TABLE RELATIONSHIPS =====")

print("""
1. users_df.id = orders_df.user_id

2. orders_df.order_id = order_items_df.order_id

3. users_df.id = order_items_df.user_id

4. products_df.id = order_items_df.product_id
""")


===== SUGGESTED TABLE RELATIONSHIPS =====

1. users_df.id = orders_df.user_id

2. orders_df.order_id = order_items_df.order_id

3. users_df.id = order_items_df.user_id

4. products_df.id = order_items_df.product_id



In [0]:
# ============================================
# Part 2: Basic Cleaning
# ============================================

from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType

# ============================================
# 7. Check Null Values in Important Columns
# ============================================

print("===== NULL VALUE CHECKS =====")

print("Users - id null count:")
print(users_df.filter(col("id").isNull()).count())

print("Orders - order_id null count:")
print(orders_df.filter(col("order_id").isNull()).count())

print("Orders - user_id null count:")
print(orders_df.filter(col("user_id").isNull()).count())

print("Order Items - order_id null count:")
print(order_items_df.filter(col("order_id").isNull()).count())

print("Order Items - product_id null count:")
print(order_items_df.filter(col("product_id").isNull()).count())

print("Products - id null count:")
print(products_df.filter(col("id").isNull()).count())


===== NULL VALUE CHECKS =====
Users - id null count:
0
Orders - order_id null count:
0
Orders - user_id null count:
0
Order Items - order_id null count:
0
Order Items - product_id null count:
0
Products - id null count:
0


In [0]:
# ============================================
# 8. Remove Rows with Null Join Keys
# ============================================

users_cleaned = users_df.dropna(subset=["id"])

orders_cleaned = orders_df.dropna(
    subset=["order_id", "user_id"]
)

order_items_cleaned = order_items_df.dropna(
    subset=["order_id", "product_id"]
)

products_cleaned = products_df.dropna(
    subset=["id"]
)

print("\n===== ROW COUNTS AFTER REMOVING NULL JOIN KEYS =====")

print("Users:", users_cleaned.count())
print("Orders:", orders_cleaned.count())
print("Order Items:", order_items_cleaned.count())
print("Products:", products_cleaned.count())



===== ROW COUNTS AFTER REMOVING NULL JOIN KEYS =====
Users: 100000
Orders: 124923
Order Items: 180952
Products: 29120


In [0]:
# ============================================
# 9. Replace Null Values in category/brand
# ============================================

products_cleaned = products_cleaned.fillna({
    "category": "unknown",
    "brand": "unknown"
})

print("\n===== CHECK NULLS AFTER REPLACEMENT =====")

products_cleaned.select("category", "brand").show(5)



===== CHECK NULLS AFTER REPLACEMENT =====
+--------+-----+
|category|brand|
+--------+-----+
|    Swim|  2XU|
|    Swim|  TYR|
|    Swim|  TYR|
|    Swim|  TYR|
|    Swim|  TYR|
+--------+-----+
only showing top 5 rows


In [0]:
# ============================================
# 10. Ensure Numeric Columns are Correct Type
# ============================================

products_cleaned = products_cleaned.withColumn(
    "retail_price",
    col("retail_price").cast(DoubleType())
)

products_cleaned = products_cleaned.withColumn(
    "cost",
    col("cost").cast(DoubleType())
)

order_items_cleaned = order_items_cleaned.withColumn(
    "sale_price",
    col("sale_price").cast(DoubleType())
)

print("\n===== UPDATED SCHEMA =====")

print("Products Schema:")
products_cleaned.printSchema()

print("Order Items Schema:")
order_items_cleaned.printSchema()


===== UPDATED SCHEMA =====
Products Schema:
root
 |-- id: integer (nullable = true)
 |-- cost: double (nullable = true)
 |-- category: string (nullable = false)
 |-- name: string (nullable = true)
 |-- brand: string (nullable = false)
 |-- retail_price: double (nullable = true)
 |-- department: string (nullable = true)
 |-- sku: string (nullable = true)
 |-- distribution_center_id: integer (nullable = true)

Order Items Schema:
root
 |-- id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- inventory_item_id: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- shipped_at: timestamp (nullable = true)
 |-- delivered_at: timestamp (nullable = true)
 |-- returned_at: timestamp (nullable = true)
 |-- sale_price: double (nullable = true)



In [0]:
# ============================================
# Part 3: Transformations
# ============================================

from pyspark.sql.functions import when, countDistinct

# ============================================
# 11. Select Required Columns
# ============================================

order_items_selected = order_items_cleaned.select(
    "order_id",
    "user_id",
    "product_id",
    "status",
    "sale_price",
    "created_at"
)

print("===== SELECTED COLUMNS =====")
order_items_selected.show(5)

===== SELECTED COLUMNS =====
+--------+-------+----------+-------+----------+-------------------+
|order_id|user_id|product_id| status|sale_price|         created_at|
+--------+-------+----------+-------+----------+-------------------+
|   10826|   8614|     13606|Shipped|       2.5|2020-06-28 00:40:17|
|   13243|  10505|     13606|Shipped|       2.5|2022-03-01 05:18:44|
|   53140|  42340|     13606|Shipped|       2.5|2021-04-11 01:31:52|
|  104681|  83704|     13606|Shipped|       2.5|2022-03-29 21:37:06|
|  117931|  94363|     13606|Shipped|       2.5|2021-02-17 09:28:17|
+--------+-------+----------+-------+----------+-------------------+
only showing top 5 rows


In [0]:
# ============================================
# 12. Create price_bucket Column
# ============================================

order_items_transformed = order_items_selected.withColumn(
    "price_bucket",
    when(col("sale_price") < 25, "Low")
    .when((col("sale_price") >= 25) & (col("sale_price") <= 75), "Medium")
    .otherwise("High")
)

print("===== PRICE BUCKET COLUMN =====")
order_items_transformed.show(5)

===== PRICE BUCKET COLUMN =====
+--------+-------+----------+-------+----------+-------------------+------------+
|order_id|user_id|product_id| status|sale_price|         created_at|price_bucket|
+--------+-------+----------+-------+----------+-------------------+------------+
|   10826|   8614|     13606|Shipped|       2.5|2020-06-28 00:40:17|         Low|
|   13243|  10505|     13606|Shipped|       2.5|2022-03-01 05:18:44|         Low|
|   53140|  42340|     13606|Shipped|       2.5|2021-04-11 01:31:52|         Low|
|  104681|  83704|     13606|Shipped|       2.5|2022-03-29 21:37:06|         Low|
|  117931|  94363|     13606|Shipped|       2.5|2021-02-17 09:28:17|         Low|
+--------+-------+----------+-------+----------+-------------------+------------+
only showing top 5 rows


In [0]:
# ============================================
# 13. Filter Records
# sale_price > 20 AND product_id IS NOT NULL
# ============================================

filtered_df = order_items_transformed.filter(
    (col("sale_price") > 20) &
    (col("product_id").isNotNull())
)

print("===== FILTERED DATA =====")
filtered_df.show(5)

===== FILTERED DATA =====
+--------+-------+----------+-------+----------+-------------------+------------+
|order_id|user_id|product_id| status|sale_price|         created_at|price_bucket|
+--------+-------+----------+-------+----------+-------------------+------------+
|    1555|   1248|     12343|Shipped|      21.0|2022-01-23 16:20:05|         Low|
|    2003|   1603|     26412|Shipped|      21.0|2021-01-02 23:38:11|         Low|
|    2163|   1743|     24521|Shipped|      21.0|2022-04-04 22:05:08|         Low|
|    2209|   1777|     24787|Shipped|      21.0|2020-12-08 02:26:58|         Low|
|    2421|   1944|     24521|Shipped|      21.0|2021-01-19 09:22:24|         Low|
+--------+-------+----------+-------+----------+-------------------+------------+
only showing top 5 rows


In [0]:
# ============================================
# 14. Find Total Counts
# ============================================

total_rows = filtered_df.count()

unique_orders = filtered_df.select("order_id") \
    .distinct() \
    .count()

unique_users = filtered_df.select("user_id") \
    .distinct() \
    .count()

unique_products = filtered_df.select("product_id") \
    .distinct() \
    .count()

print("===== SUMMARY METRICS =====")

print(f"Total Rows: {total_rows}")
print(f"Unique Orders: {unique_orders}")
print(f"Unique Users: {unique_users}")
print(f"Unique Products: {unique_products}")

===== SUMMARY METRICS =====
Total Rows: 144100
Unique Orders: 106092
Unique Users: 71873
Unique Products: 23044


In [0]:
# ============================================
# Part 4: Joins
# ============================================

from pyspark.sql.functions import broadcast

In [0]:
# ============================================
# 15. Join users.id with orders.user_id
# ============================================

users_orders_df = users_cleaned.join(
    orders_cleaned,
    users_cleaned.id == orders_cleaned.user_id,
    "inner"
)

print("===== USERS + ORDERS JOINED =====")
users_orders_df.show(5)


===== USERS + ORDERS JOINED =====
+-----+----------+---------+--------------------+---+------+-----+--------------------+-----------+-----------+-------+------------+------------+--------------+-------------------+--------+-------+---------+------+-------------------+-----------+-------------------+-------------------+-----------+
|   id|first_name|last_name|               email|age|gender|state|      street_address|postal_code|       city|country|    latitude|   longitude|traffic_source|         created_at|order_id|user_id|   status|gender|         created_at|returned_at|         shipped_at|       delivered_at|num_of_item|
+-----+----------+---------+--------------------+---+------+-----+--------------------+-----------+-----------+-------+------------+------------+--------------+-------------------+--------+-------+---------+------+-------------------+-----------+-------------------+-------------------+-----------+
|44262|   Michael|  Sanchez|michaelsanchez@ex...| 48|     M|  Mie|   

In [0]:
# ============================================
# 16. Join orders.order_id with order_items.order_id
# ============================================

users_orders_items_df = users_orders_df.join(
    order_items_transformed,
    users_orders_df.order_id == order_items_transformed.order_id,
    "inner"
)

print("===== USERS + ORDERS + ORDER_ITEMS JOINED =====")
users_orders_items_df.show(5)

===== USERS + ORDERS + ORDER_ITEMS JOINED =====
+-----+----------+---------+--------------------+---+------+-----+--------------------+-----------+-----------+-------+------------+------------+--------------+-------------------+--------+-------+---------+------+-------------------+-----------+-------------------+-------------------+-----------+--------+-------+----------+---------+------------------+-------------------+------------+
|   id|first_name|last_name|               email|age|gender|state|      street_address|postal_code|       city|country|    latitude|   longitude|traffic_source|         created_at|order_id|user_id|   status|gender|         created_at|returned_at|         shipped_at|       delivered_at|num_of_item|order_id|user_id|product_id|   status|        sale_price|         created_at|price_bucket|
+-----+----------+---------+--------------------+---+------+-----+--------------------+-----------+-----------+-------+------------+------------+--------------+--------------

In [0]:
# ============================================
# 17. Join order_items.product_id with products.id
# Using Broadcast Join (small dimension table optimization)
# ============================================

final_df = users_orders_items_df.join(
    broadcast(products_cleaned),
    users_orders_items_df.product_id == products_cleaned.id,
    "inner"
)

print("===== FINAL JOIN COMPLETED =====")

===== FINAL JOIN COMPLETED =====


In [0]:
# ============================================
# 18. Final DataFrame = final_df
# ============================================

# final_df already created above


# ============================================
# 19. Show Schema, Row Count, and Sample Records
# ============================================

print("===== FINAL DATAFRAME SCHEMA =====")
final_df.printSchema()

print("===== FINAL DATAFRAME ROW COUNT =====")
print(final_df.count())

print("===== SAMPLE RECORDS FROM final_df =====")
final_df.show(10)

===== FINAL DATAFRAME SCHEMA =====
root
 |-- id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- state: string (nullable = true)
 |-- street_address: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- traffic_source: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- returned_at: timestamp (nullable = true)
 |-- shipped_at: timestamp (nullable = true)
 |-- delivered_at: timestamp (nullable = true)
 |-- num_of_item: 

In [0]:
# ============================================
# Part 5: Aggregations
# ============================================

from pyspark.sql.functions import count, sum, avg, round

# ============================================
# 20. Top 10 Product Categories by Number of Items Sold
# ============================================

top_categories = final_df.groupBy("category") \
    .agg(count("product_id").alias("items_sold")) \
    .orderBy(col("items_sold").desc()) \
    .limit(10)

print("===== TOP 10 PRODUCT CATEGORIES BY ITEMS SOLD =====")
top_categories.show()


===== TOP 10 PRODUCT CATEGORIES BY ITEMS SOLD =====
+--------------------+----------+
|            category|items_sold|
+--------------------+----------+
|           Intimates|     13253|
|               Jeans|     12604|
|         Tops & Tees|     11906|
|Fashion Hoodies &...|     11878|
|                Swim|     11373|
|      Sleep & Lounge|     11066|
|            Sweaters|     11037|
|              Shorts|     11002|
|         Accessories|      9890|
|   Outerwear & Coats|      9040|
+--------------------+----------+



In [0]:
# ============================================
# 21. Top 10 Brands by Total Sales
# ============================================

top_brands = final_df.groupBy("brand") \
    .agg(
        round(sum("sale_price"), 2).alias("total_sales")
    ) \
    .orderBy(col("total_sales").desc()) \
    .limit(10)

print("===== TOP 10 BRANDS BY TOTAL SALES =====")
top_brands.show()

===== TOP 10 BRANDS BY TOTAL SALES =====
+-----------------+-----------+
|            brand|total_sales|
+-----------------+-----------+
|     Calvin Klein|  198826.17|
|           Diesel|   197385.5|
|         Carhartt|  178058.31|
|    True Religion|   177432.7|
|7 For All Mankind|  169054.31|
|   Tommy Hilfiger|   126174.9|
|           Volcom|  111350.97|
|   The North Face|  108887.59|
|         Columbia|  103321.14|
|      Joe's Jeans|   102971.7|
+-----------------+-----------+



In [0]:
# ============================================
# 22. Top 10 States by Number of Orders
# ============================================

# Use users_orders_df which has state and a single order_id column
top_states_orders = users_orders_df.groupBy("state") \
    .agg(
        countDistinct("order_id").alias("total_orders")
    ) \
    .orderBy(col("total_orders").desc()) \
    .limit(10)

print("===== TOP 10 STATES BY NUMBER OF ORDERS =====")
top_states_orders.show()

===== TOP 10 STATES BY NUMBER OF ORDERS =====
+----------+------------+
|     state|total_orders|
+----------+------------+
| Guangdong|        6679|
|   England|        5086|
|California|        4598|
|  Shanghai|        3109|
|     Texas|        3033|
| São Paulo|        2689|
|   Beijing|        2627|
|     Hebei|        2539|
|  Zhejiang|        2538|
|   Jiangsu|        2354|
+----------+------------+



In [0]:
# ============================================
# 23. Top 10 States by Total Sales Value
# ============================================

top_states_sales = final_df.groupBy("state") \
    .agg(
        round(sum("sale_price"), 2).alias("total_sales")
    ) \
    .orderBy(col("total_sales").desc()) \
    .limit(10)

print("===== TOP 10 STATES BY TOTAL SALES VALUE =====")
top_states_sales.show()

===== TOP 10 STATES BY TOTAL SALES VALUE =====
+----------+-----------+
|     state|total_sales|
+----------+-----------+
| Guangdong|  569099.26|
|   England|  434056.52|
|California|  396677.51|
|     Texas|  261580.15|
|  Shanghai|  261549.47|
| São Paulo|  232593.85|
|   Beijing|  227643.94|
|     Hebei|  222767.15|
|  Zhejiang|  219429.18|
|   Jiangsu|  194777.63|
+----------+-----------+



In [0]:
# ============================================
# 24. Average Sale Price by Category
# ============================================

avg_sale_price = final_df.groupBy("category") \
    .agg(
        round(avg("sale_price"), 2).alias("avg_sale_price")
    ) \
    .orderBy(col("avg_sale_price").desc())

print("===== AVERAGE SALE PRICE BY CATEGORY =====")
avg_sale_price.show()

===== AVERAGE SALE PRICE BY CATEGORY =====
+--------------------+--------------+
|            category|avg_sale_price|
+--------------------+--------------+
|   Outerwear & Coats|        143.91|
| Suits & Sport Coats|        126.07|
|               Suits|        116.63|
|               Jeans|          99.0|
|   Blazers & Jackets|         89.55|
|       Clothing Sets|         88.12|
|             Dresses|         84.97|
|            Sweaters|          76.4|
|               Pants|         59.83|
|                Swim|         57.26|
|      Pants & Capris|         54.86|
|Fashion Hoodies &...|         53.97|
|              Skirts|         51.85|
|           Maternity|         51.14|
|              Active|         50.06|
|      Sleep & Lounge|         49.65|
|              Shorts|          45.8|
| Jumpsuits & Rompers|         45.74|
|         Accessories|         42.16|
|         Tops & Tees|         41.38|
+--------------------+--------------+
only showing top 20 rows


In [0]:
# ============================================
# 25. Count of Completed, Returned, and Cancelled Items
# ============================================

# Use order_items_transformed which has a single status column
status_counts = order_items_transformed.groupBy("status") \
    .agg(
        count("*").alias("item_count")
    ) \
    .orderBy(col("item_count").desc())

print("===== ITEM STATUS COUNTS =====")
status_counts.show()

===== ITEM STATUS COUNTS =====
+----------+----------+
|    status|item_count|
+----------+----------+
|   Shipped|     54681|
|  Complete|     45010|
|Processing|     35941|
| Cancelled|     27133|
|  Returned|     18187|
+----------+----------+



In [0]:
# ============================================
# Part 6: Business Questions
# ============================================

from pyspark.sql.functions import countDistinct

# ============================================
# 26. Which State Generated the Highest Sales?
# ============================================

highest_sales_state = final_df.groupBy("state") \
    .agg(
        round(sum("sale_price"), 2).alias("total_sales")
    ) \
    .orderBy(col("total_sales").desc())

print("===== STATE WITH HIGHEST SALES =====")
highest_sales_state.show(1)


===== STATE WITH HIGHEST SALES =====
+---------+-----------+
|    state|total_sales|
+---------+-----------+
|Guangdong|  569099.26|
+---------+-----------+
only showing top 1 row


In [0]:
# ============================================
# 27. Which Category Generated the Highest Revenue?
# ============================================

highest_revenue_category = final_df.groupBy("category") \
    .agg(
        round(sum("sale_price"), 2).alias("total_revenue")
    ) \
    .orderBy(col("total_revenue").desc())

print("===== CATEGORY WITH HIGHEST REVENUE =====")
highest_revenue_category.show(1)


===== CATEGORY WITH HIGHEST REVENUE =====
+-----------------+-------------+
|         category|total_revenue|
+-----------------+-------------+
|Outerwear & Coats|   1300914.26|
+-----------------+-------------+
only showing top 1 row


In [0]:
# ============================================
# 28. Which Brand Appears Most Frequently?
# ============================================

most_frequent_brand = final_df.groupBy("brand") \
    .agg(
        count("*").alias("brand_count")
    ) \
    .orderBy(col("brand_count").desc())

print("===== MOST FREQUENT BRAND =====")
most_frequent_brand.show(1)


===== MOST FREQUENT BRAND =====
+---------+-----------+
|    brand|brand_count|
+---------+-----------+
|Allegra K|       6262|
+---------+-----------+
only showing top 1 row


In [0]:
# ============================================
# 29. Which Gender Placed More Orders?
# ============================================

# Join users and orders, selecting specific columns to avoid ambiguity
gender_orders = users_cleaned.join(
    orders_cleaned.select("order_id", "user_id"),
    users_cleaned.id == orders_cleaned.user_id,
    "inner"
).groupBy(users_cleaned.gender) \
    .agg(
        countDistinct("order_id").alias("total_orders")
    ) \
    .orderBy(col("total_orders").desc())

print("===== ORDERS BY GENDER =====")
gender_orders.show()

===== ORDERS BY GENDER =====
+------+------------+
|gender|total_orders|
+------+------------+
|     M|       62525|
|     F|       62398|
+------+------------+



In [0]:
# ============================================
# 30. Which Orders Contain More Than One Item?
# ============================================

# Use order_items_transformed which has a single order_id column
multi_item_orders = order_items_transformed.groupBy("order_id") \
    .agg(
        count("product_id").alias("item_count")
    ) \
    .filter(col("item_count") > 1) \
    .orderBy(col("item_count").desc())

print("===== ORDERS WITH MORE THAN ONE ITEM =====")
multi_item_orders.show(10)

===== ORDERS WITH MORE THAN ONE ITEM =====
+--------+----------+
|order_id|item_count|
+--------+----------+
|    1289|         4|
|   41266|         4|
|  101478|         4|
|   19424|         4|
|  100065|         4|
|   91407|         4|
|   56273|         4|
|   25719|         4|
|   60340|         4|
|  120959|         4|
+--------+----------+
only showing top 10 rows


In [0]:
# ============================================
# 31. Which Categories Have the Highest Average Sale Price?
# ============================================

highest_avg_category = final_df.groupBy("category") \
    .agg(
        round(avg("sale_price"), 2).alias("avg_sale_price")
    ) \
    .orderBy(col("avg_sale_price").desc())

print("===== CATEGORIES WITH HIGHEST AVERAGE SALE PRICE =====")
highest_avg_category.show(10)

===== CATEGORIES WITH HIGHEST AVERAGE SALE PRICE =====
+-------------------+--------------+
|           category|avg_sale_price|
+-------------------+--------------+
|  Outerwear & Coats|        143.91|
|Suits & Sport Coats|        126.07|
|              Suits|        116.63|
|              Jeans|          99.0|
|  Blazers & Jackets|         89.55|
|      Clothing Sets|         88.12|
|            Dresses|         84.97|
|           Sweaters|          76.4|
|              Pants|         59.83|
|               Swim|         57.26|
+-------------------+--------------+
only showing top 10 rows


In [0]:
# ============================================
# Part 7: Cache / Persist
# ============================================

from pyspark.storagelevel import StorageLevel

# ============================================
# 32. Cache final_df
# ============================================

# Note: Caching is not supported on serverless compute
# The DataFrame will be recomputed as needed

print("===== CACHING NOT AVAILABLE ON SERVERLESS COMPUTE =====")

===== CACHING NOT AVAILABLE ON SERVERLESS COMPUTE =====


In [0]:
# ============================================
# 33A. Category-wise Revenue Analysis
# ============================================

category_revenue = final_df.groupBy("category") \
    .agg(
        round(sum("sale_price"), 2).alias("total_revenue")
    ) \
    .orderBy(col("total_revenue").desc())

print("===== CATEGORY-WISE REVENUE =====")
category_revenue.show(10)


===== CATEGORY-WISE REVENUE =====
+--------------------+-------------+
|            category|total_revenue|
+--------------------+-------------+
|   Outerwear & Coats|   1300914.26|
|               Jeans|   1247804.99|
|            Sweaters|    843207.47|
|                Swim|     651241.3|
| Suits & Sport Coats|    645624.85|
|Fashion Hoodies &...|    641006.33|
|      Sleep & Lounge|    549430.39|
|              Shorts|    503854.08|
|         Tops & Tees|    492713.88|
|             Dresses|    461990.43|
+--------------------+-------------+
only showing top 10 rows


In [0]:
# ============================================
# 33B. State-wise Order Count Analysis
# ============================================

# Use users_orders_df which has a single order_id column
state_order_count = users_orders_df.groupBy("state") \
    .agg(
        countDistinct("order_id").alias("order_count")
    ) \
    .orderBy(col("order_count").desc())

print("===== STATE-WISE ORDER COUNT =====")
state_order_count.show(10)

===== STATE-WISE ORDER COUNT =====
+----------+-----------+
|     state|order_count|
+----------+-----------+
| Guangdong|       6679|
|   England|       5086|
|California|       4598|
|  Shanghai|       3109|
|     Texas|       3033|
| São Paulo|       2689|
|   Beijing|       2627|
|     Hebei|       2539|
|  Zhejiang|       2538|
|   Jiangsu|       2354|
+----------+-----------+
only showing top 10 rows


In [0]:
# ============================================
# 34. Why Caching Helps
# ============================================

print("""
===== WHY CACHING HELPS =====

1. Caching stores the joined DataFrame in memory, reducing the need to recompute expensive joins repeatedly.

2. When the same DataFrame is reused for multiple analyses, Spark can access cached data faster,
   improving overall performance and reducing execution time.

3. Caching is especially useful for large datasets and iterative analytics workflows.
""")


===== WHY CACHING HELPS =====

1. Caching stores the joined DataFrame in memory, reducing the need to recompute expensive joins repeatedly.

2. When the same DataFrame is reused for multiple analyses, Spark can access cached data faster,
   improving overall performance and reducing execution time.

3. Caching is especially useful for large datasets and iterative analytics workflows.



In [0]:
# ============================================
# Part 8: Broadcast Join
# ============================================

from pyspark.sql.functions import broadcast

# ============================================
# 35. Apply Broadcast Join
# ============================================

# Joining transactional data with smaller products table
broadcast_join_df = users_orders_items_df.join(
    broadcast(products_cleaned),
    users_orders_items_df.product_id == products_cleaned.id,
    "inner"
)

print("===== BROADCAST JOIN COMPLETED =====")

broadcast_join_df.show(5)

print("Row Count:", broadcast_join_df.count())


===== BROADCAST JOIN COMPLETED =====
+-----+----------+---------+--------------------+---+------+-----+--------------------+-----------+-----------+-------+------------+------------+--------------+-------------------+--------+-------+---------+------+-------------------+-----------+-------------------+-------------------+-----------+--------+-------+----------+---------+------------------+-------------------+------------+-----+------------------+--------------------+--------------------+-----------------+------------------+----------+--------------------+----------------------+
|   id|first_name|last_name|               email|age|gender|state|      street_address|postal_code|       city|country|    latitude|   longitude|traffic_source|         created_at|order_id|user_id|   status|gender|         created_at|returned_at|         shipped_at|       delivered_at|num_of_item|order_id|user_id|product_id|   status|        sale_price|         created_at|price_bucket|   id|              cost|  

In [0]:
# ============================================
# 36. Explanation of Broadcast Join
# ============================================

print("""
===== WHAT IS A BROADCAST JOIN? =====

1. A broadcast join sends the smaller DataFrame to all worker nodes in the cluster,
   avoiding large data shuffling during joins.

2. It improves performance because Spark can perform the join locally on each executor
   instead of transferring large amounts of data across the network.

3. Broadcast joins are especially useful when one table (like products) is relatively small
   compared to large transactional tables.
""")


===== WHAT IS A BROADCAST JOIN? =====

1. A broadcast join sends the smaller DataFrame to all worker nodes in the cluster,
   avoiding large data shuffling during joins.

2. It improves performance because Spark can perform the join locally on each executor
   instead of transferring large amounts of data across the network.

3. Broadcast joins are especially useful when one table (like products) is relatively small
   compared to large transactional tables.



In [0]:
# ============================================
# Part 9: Repartitioning and Output
# ============================================

# ============================================
# 37. Repartition final_df by state
# ============================================

repartitioned_df = final_df.repartition("state")

print("===== DATAFRAME REPARTITIONED BY STATE =====")
print("Data repartitioned by 'state' column for optimized downstream operations.")

===== DATAFRAME REPARTITIONED BY STATE =====
Data repartitioned by 'state' column for optimized downstream operations.


In [0]:
# ============================================
# 38. Write Repartitioned DataFrame in Parquet Format
# ============================================

output_path = "/Volumes/workspace/default/ecommerce/final_df_parquet"

# Create unique column names by appending suffixes to duplicates
columns = repartitioned_df.columns
unique_column_names = []
seen = {}

for col_name in columns:
    if col_name not in seen:
        unique_column_names.append(col_name)
        seen[col_name] = 1
    else:
        seen[col_name] += 1
        unique_column_names.append(f"{col_name}_{seen[col_name]}")

repartitioned_df_clean = repartitioned_df.toDF(*unique_column_names)

repartitioned_df_clean.write \
    .mode("overwrite") \
    .parquet(output_path)

print("===== PARQUET FILE WRITTEN SUCCESSFULLY =====")

print(f"Output Location: {output_path}")

===== PARQUET FILE WRITTEN SUCCESSFULLY =====
Output Location: /Volumes/workspace/default/ecommerce/final_df_parquet


In [0]:
# ============================================
# 39. Why Repartitioning Helps
# ============================================

print("""
===== WHY REPARTITIONING HELPS =====

1. Repartitioning distributes data evenly across partitions, improving parallel processing
   and query performance in Spark.

2. Partitioning by state helps because many business analyses are region-based,
   such as state-wise sales, customer activity, and order trends.

3. It also improves efficiency when filtering or querying data for a specific state,
   since Spark can read only relevant partitions.
""")


===== WHY REPARTITIONING HELPS =====

1. Repartitioning distributes data evenly across partitions, improving parallel processing
   and query performance in Spark.

2. Partitioning by state helps because many business analyses are region-based,
   such as state-wise sales, customer activity, and order trends.

3. It also improves efficiency when filtering or querying data for a specific state,
   since Spark can read only relevant partitions.



In [0]:
# ============================================
# Optional Bonus Task
# Monthly Order Count using created_at
# ============================================

from pyspark.sql.functions import to_timestamp, date_format

# ============================================
# Convert created_at to Timestamp
# ============================================

# Use orders_cleaned which has a single created_at column
monthly_orders_df = orders_cleaned.withColumn(
    "created_timestamp",
    to_timestamp(col("created_at"))
)

# ============================================
# Extract Month-Year
# ============================================

monthly_orders_df = monthly_orders_df.withColumn(
    "order_month",
    date_format(col("created_timestamp"), "yyyy-MM")
)

# ============================================
# Count Orders per Month
# ============================================

monthly_order_count = monthly_orders_df.groupBy("order_month") \
    .agg(
        count("order_id").alias("order_count")
    ) \
    .orderBy(col("order_month"))

print("===== MONTHLY ORDER COUNT =====")
monthly_order_count.show()

===== MONTHLY ORDER COUNT =====
+-----------+-----------+
|order_month|order_count|
+-----------+-----------+
|    2019-01|         30|
|    2019-02|        101|
|    2019-03|        178|
|    2019-04|        233|
|    2019-05|        312|
|    2019-06|        430|
|    2019-07|        527|
|    2019-08|        590|
|    2019-09|        670|
|    2019-10|        802|
|    2019-11|        873|
|    2019-12|        993|
|    2020-01|       1097|
|    2020-02|       1087|
|    2020-03|       1302|
|    2020-04|       1380|
|    2020-05|       1485|
|    2020-06|       1576|
|    2020-07|       1799|
|    2020-08|       1894|
+-----------+-----------+
only showing top 20 rows
